# 01-Foundations

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

# 18-Data-Acquisition-Web-Scraping



[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)

[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

In [1]:
# --- Global Notebook Setup ---
import os
import sys
import math
import time
import random
import json
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError

# Apply the standard course style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'lines.markersize': 6
})
%config InlineBackend.figure_format = 'retina'  # High-res plots

np.set_printoptions(suppress=True, linewidth=120, precision=4)
warnings.filterwarnings('ignore', category=FutureWarning)
print("Environment initialized.")

Environment initialized.


### Table of Contents
1. [The Lens: Web Scraping as Data Excavation](#The-Lens:-Web-Scraping-as-Data-Excavation)
2. [The Ethics and Legality of Scraping](#The-Ethics-and-Legality-of-Scraping)
3. [Static Scraping with `requests` and `BeautifulSoup`](#Static-Scraping-with-requests-and-BeautifulSoup)
    - [Parsing HTML Structure](#Parsing-HTML-Structure)
    - [Extracting Data into Pandas](#Extracting-Data-into-Pandas)
4. [Dynamic Scraping with `playwright`](#Dynamic-Scraping-with-playwright)
    - [Why Not Selenium?](#Why-Not-Selenium?)
    - [Handling JavaScript and Interaction](#Handling-JavaScript-and-Interaction)
5. [Summary](#Summary)
6. [Exercises](#Exercises)

# The Lens

While APIs provide a clean pipeline to data, the vast majority of the world's information exists as unstructured HTML on websites. Web scraping is the digital equivalent of archaeological excavation: it involves carefully digging through layers of presentation code (HTML, CSS, JavaScript) to extract the valuable artifacts (data) buried underneath.

This notebook focuses specifically on the techniques for this excavation. We will move beyond the simple examples in the introductory chapter to tackle more realistic scenarios, including navigating complex HTML structures and handling dynamic, JavaScript-heavy sites that resist simple request-based scraping.

### The Ethics and Legality of Scraping

Before writing a single line of code, you must understand the rules of the road. 

1.  **Check `robots.txt`**: Always checks `domain.com/robots.txt`. It dictates which parts of a site are off-limits to bots.
2.  **Rate Limiting**: Never slam a server with hundreds of requests per second. Use `time.sleep()` to add delays between requests.
3.  **Terms of Service**: Read the site's ToS. Scraping public data is generally legal in many jurisdictions (e.g., the hiQ vs. LinkedIn ruling in the US), but violating ToS can lead to your IP being banned.
4.  **Identify Yourself**: Set a custom `User-Agent` string in your headers that identifies your bot and provides a way to contact you (e.g., an email address).

### Static Scraping with `requests` and `BeautifulSoup`

For websites that serve their content as static HTML (i.e., the data is present in the page source when you right-click -> "View Source"), the combination of `requests` and `BeautifulSoup` is the standard toolset. It is fast, lightweight, and robust.

In [2]:
def fetch_page(url):
    """Fetches a page with a polite User-Agent."""
    headers = {
        'User-Agent': 'EconomicResearchBot/1.0 (contact: student@university.edu)'
    }
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

url = 'http://quotes.toscrape.com/'
html_content = fetch_page(url)

if html_content:
    soup = BeautifulSoup(html_content, 'html.parser')
    print(f"Page Title: {soup.title.text}")
    print("\nFirst 500 characters of HTML source:")
    print(soup.prettify()[:500])

Page Title: Quotes to Scrape

First 500 characters of HTML source:
<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scrape
      </a>
     </h1>
    </div>
    <div class="col-md-4">
     <p>
      <a href="/login">
   


#### Extracting Data into Pandas

The goal of scraping is usually to create a structured dataset. Let's scrape the quotes, authors, and tags from the page and load them into a Pandas DataFrame.

In [3]:
if html_content:
    quotes_data = []
    # Find all div elements with class 'quote'
    quote_divs = soup.find_all('div', class_='quote')
    
    for div in quote_divs:
        text = div.find('span', class_='text').get_text(strip=True)
        author = div.find('small', class_='author').get_text(strip=True)
        # Tags are in a meta tag or listed links
        tags = [tag.get_text(strip=True) for tag in div.find_all('a', class_='tag')]
        
        quotes_data.append({
            'quote': text,
            'author': author,
            'tags': tags
        })
    
    df_quotes = pd.DataFrame(quotes_data)
    print("Scraped Data:")
    display(df_quotes.head())

Scraped Data:


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"


### Dynamic Scraping with `playwright`

Many modern sites (like Twitter/X, infinite scroll pages, or complex dashboards) do not serve data in the initial HTML. Instead, they send a skeleton page and use JavaScript to fetch and render the data. `requests` cannot see this data. We need a tool that controls a real web browser.

#### Why Not Selenium?
Selenium was the industry standard for years, but it was designed for testing web apps, not scraping. It is often slow and flaky. **Playwright** (by Microsoft) is a modern alternative that is faster, more reliable, and has better support for modern web features like "waiting for network idle" or intercepting API calls.

In [4]:
def scrape_js_site():
    """Scrapes a site that requires JS execution."""
    # Note: We use the JS version of the site
    url = 'http://quotes.toscrape.com/js/'
    
    try:
        with sync_playwright() as p:
            # Launch browser (headless=True is standard for scraping)
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            
            print(f"Navigating to {url}...")
            page.goto(url)
            
            # Critical: Wait for the content to appear.
            # The 'div.quote' elements are added by JS after page load.
            page.wait_for_selector('div.quote')
            
            # We can now get the fully rendered HTML
            content = page.content()
            browser.close()
            
            # Parse with BeautifulSoup as before
            soup_js = BeautifulSoup(content, 'html.parser')
            js_quotes = soup_js.find_all('div', class_='quote')
            print(f"Successfully found {len(js_quotes)} quotes on the JS-rendered page.")
            
    except Exception as e:
        print(f"Playwright failed (likely environment issue in this notebook context): {e}")

scrape_js_site()

Playwright failed (likely environment issue in this notebook context): It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.


# Summary

Web scraping is a powerful skill for acquiring novel data. 
- **Static Sites:** Use `requests` + `BeautifulSoup`. It's fast and simple.
- **Dynamic Sites:** Use `playwright` to render JavaScript.
- **Ethics:** Always respect `robots.txt` and rate limits.
- **Workflow:** Fetch raw HTML -> Parse to Structure -> Save to CSV/Parquet -> Analyze.

### Exercises

1.  **Scraping a Table:** Visit `https://www.worldometers.info/world-population/population-by-country/`. Write a script using `requests` and `pd.read_html` (which uses BeautifulSoup under the hood) to scrape the main population table into a clean Pandas DataFrame.

2.  **Pagination Logic:** Modify the quotes scraper to handle pagination. The bottom of the page has a "Next" button. Write a loop that scrapes the current page, finds the link to the next page, and continues until there are no more pages.

3.  **Playwright Interaction:** Use Playwright to go to `http://quotes.toscrape.com/search`. This page requires you to select a tag from a dropdown or type it in. Write a script that uses Playwright to type "love" into the tag filter, clicks search, and then scrapes the resulting quotes.